In [31]:
from transformers import pipeline, logging

# 1. Silence the library's internal warnings
logging.set_verbosity_error()

# 2. Load the model
pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto")

# 3. Your Prompt (The Few-Shot example strategy)
prompt = """<|system|>
You are a professional leadership coach who writes in concise paragraphs. 
You never use lists, bullet points, or numbered sections.
<|user|>
Explain the importance of emotional intelligence in leadership.
<|assistant|>
Emotional intelligence is the cornerstone of effective leadership, as it allows managers to navigate complex social dynamics with empathy and clarity. By mastering their own emotions, leaders create a foundation of trust that enables them to resolve conflicts constructively and keep teams aligned during challenging periods. Ultimately, this skill fosters a resilient organizational culture where members feel valued and understood, directly driving long-term productivity and team retention.

<|user|>
Explain the importance of clear communication in leadership.
<|assistant|>
"""

# 4. Generate (No max_length needed here, it's cleaner)
output = pipe(
    prompt, 
    max_new_tokens=250, 
    temperature=0.1, 
    do_sample=False
)

# 5. Show Result
print(output[0]['generated_text'].split("<|assistant|>")[-1])

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4797.55it/s]



Clear communication is essential for effective leadership, as it enables team members to understand the goals and expectations of the organization, and to collaborate effectively to achieve them. By communicating clearly, leaders can ensure that everyone is on the same page and that decisions are made with the best interests of the organization in mind. This leads to increased trust, teamwork, and a more productive work environment.

Additionally, clear communication can help to prevent misunderstandings and conflicts, as it allows individuals to understand the context and intent behind decisions. This can help to resolve conflicts and build stronger relationships between team members, ultimately leading to a more cohesive and productive organization.

Overall, clear communication is a critical component of effective leadership, as it enables individuals to work together towards a common goal and fosters a culture of trust and collaboration.


In [27]:
# New cell: Install the necessary dependencies
!pip install transformers torch accelerate


In [9]:
from datasets import load_dataset

# 1. Load a high-quality "Instruction-Tuning" dataset (Alpaca style)
# We take 500 rows to ensure it fits in memory and trains quickly
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:500]")

# 2. Format into the structure your model understands
def format_data(example):
    # This matches the Instruction/Answer structure we discussed
    return f"Instruction: {example['instruction']}\nAnswer: {example['output']}\n\n"

# 3. Save to training_data.txt
with open("training_data.txt", "w", encoding="utf-8") as f:
    for ex in dataset:
        f.write(format_data(ex))

print("Pro-level dataset created with 500 high-quality examples!")

Pro-level dataset created with 500 high-quality examples!


In [8]:
!pip install datasets

In [13]:
# Save to a fresh folder name to avoid the lock
save_path = "./gpt2_final_model_v4"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved successfully to {save_path}!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.98it/s]

Model saved successfully to ./gpt2_final_model_v4!


In [17]:
from transformers import pipeline

# 1. Path to your new saved model
model_path = "./gpt2_final_model_v4"  # Update this if you used a different name

# 2. Initialize the pipeline
# We load both model and tokenizer from the saved folder
custom_ai = pipeline(
    "text-generation", 
    model=model_path, 
    tokenizer=model_path
)

# 3. Test it with a prompt from your new pro training data
# Force the AI to be concise using instructions inside the prompt
prompt = """Instruction: Explain the importance of emotional intelligence in leadership. 
Constraint: Answer in exactly three sentences. Do not mention depression, history, or external advice. Focus only on professional leadership.
Answer:"""

output = custom_ai(
    prompt, 
    max_new_tokens=100, # Keep this tight so it can't ramble
    temperature=0.1,    # Keep this very low
    repetition_penalty=2.5, # Strict penalty
    do_sample=False,
    pad_token_id=50256
)

print("\n--- AI OUTPUT ---")
print(output[0]['generated_text'])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7914.39it/s]
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- AI OUTPUT ---
Instruction: Explain the importance of emotional intelligence in leadership. 
Constraint: Answer in exactly three sentences. Do not mention depression, history, or external advice. Focus only on professional leadership.
Answer: The key to success is a strong sense that you are passionate about your job and strive for excellence at every step along its way; this can be achieved through hard work alone rather than by relying solely upon one person's abilities or accomplishments as motivation (i-II). This type will lead people towards greater goals while also providing them with an opportunity toward achieving their own personal growth within themselves without compromising others' ability/talents!


In [5]:
trainer.train()

e:\ShadowFox\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.184540
100,3.011094
150,3.023332
200,2.945036
250,2.877505
300,2.911615
350,2.938234
400,2.850399
450,2.798873
500,2.809596


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]


TrainOutput(global_step=626, training_loss=2.902151942633973, metrics={'train_runtime': 1337.1663, 'train_samples_per_second': 1.873, 'train_steps_per_second': 0.468, 'total_flos': 34873300224000.0, 'train_loss': 2.902151942633973, 'epoch': 1.0})

In [4]:
from transformers import TrainingArguments, Trainer

# Updated for larger datasets
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-pro",
    num_train_epochs=1,               # 1 epoch is plenty for 500 complex examples
    per_device_train_batch_size=1,    # Small batch size to save memory
    gradient_accumulation_steps=4,    # This acts like a batch size of 4
    save_steps=100,
    logging_steps=50,
    report_to="none"
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("Trainer setup configured for deeper training (10 epochs)!")

Trainer setup configured for deeper training (10 epochs)!


In [3]:
import torch
from transformers import DataCollatorForLanguageModeling

# 1. Read the text file directly using plain Python
with open("training_data.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Clean out any accidental blank lines
text_lines = [line.strip() for line in lines if line.strip()]

# 2. Tokenize the text right here in the main notebook process
# (We add a fallback just in case the notebook memory cleared it)
if 'tokenizer' not in globals():
    from transformers import GPT2Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizing data...")
tokenized_outputs = tokenizer(text_lines, truncation=True, max_length=128, padding=False)

# 3. Format it cleanly into a dataset list that the Trainer expects
train_dataset = [
    {"input_ids": tokenized_outputs["input_ids"][i]}
    for i in range(len(tokenized_outputs["input_ids"]))
]

# 4. Set up the standard data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False
)

print(f"Success! Loaded {len(train_dataset)} text segments. Ready for training!")

Tokenizing data...
Success! Loaded 2504 text segments. Ready for training!


In [2]:
def generate_text(prompt, max_length=50, temperature=0.7):
    # 1. Tokenize the input text
    inputs = tokenizer(prompt, return_tensors="pt")
    
    start_time = time.time()
    
    # 2. Generate text sequence
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        temperature=temperature,
        do_sample=True,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id
    )
    
    end_time = time.time()
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generation_time = end_time - start_time
    
    return generated_text, generation_time

# Run a quick generation test
test_prompt = "The future of artificial intelligence in software engineering is"
print(f"Prompt: {test_prompt}\n")
print("-" * 40)

result, time_taken = generate_text(test_prompt, max_length=60, temperature=0.7)
print(f"Generated Text:\n{result}\n")
print("-" * 40)
print(f"Time taken: {time_taken:.2f} seconds")

Prompt: The future of artificial intelligence in software engineering is

----------------------------------------
Generated Text:
The future of artificial intelligence in software engineering is uncertain, even without a clear roadmap. While the future of artificial intelligence in software engineering is uncertain, and while the prospects of it have not been announced, there is still a significant amount of research and engineering work that needs to be done in order to fully

----------------------------------------
Time taken: 2.28 seconds


In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import matplotlib.pyplot as plt
import time

print("Loading GPT-2 Model and Tokenizer...")

# Load pre-trained model and tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Set the pad token to avoid warnings
tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully! Ready for generation.")

e:\ShadowFox\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading GPT-2 Model and Tokenizer...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11056.12it/s]


Model loaded successfully! Ready for generation.
